# Notebook 01: Chuẩn bị Dữ liệu & Đồng bộ lên Amazon S3 (Tuần 2)
**Môn học:** Cloud - Đợt 1 - 2026-2027 · GVHD: Huỳnh Xuân Phụng  
**Mục tiêu:** Tạo S3 Bucket, gắn thẻ Tag quy chuẩn và tải dataset ($\le$ 2.000 ảnh) lên S3.

In [ ]:
import os
import boto3
import sagemaker

# 1. Khởi tạo phiên làm việc và lấy thông tin Account ID
session = boto3.session.Session()
region = session.region_name or 'us-east-1'
account_id = boto3.client('sts').get_caller_identity()['Account']

bucket_name = f"sagemaker-plant-leaf-{account_id}"
print(f"Account ID : {account_id}")
print(f"Region     : {region}")
print(f"S3 Bucket  : {bucket_name}")

In [ ]:
# 2. Tạo S3 Bucket trên Region us-east-1
s3 = boto3.client('s3', region_name=region)
try:
    if region == 'us-east-1':
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': region})
    print(f"[THÀNH CÔNG] Đã tạo hoặc kết nối S3 Bucket: {bucket_name}")
except Exception as e:
    print(f"Thông báo: {e}")

In [ ]:
# 3. Gắn thẻ Tag bắt buộc theo quy chế môn học
s3.put_bucket_tagging(
    Bucket=bucket_name,
    Tagging={
        'TagSet': [
            {'Key': 'Project', 'Value': 'PlantLeafClassification'},
            {'Key': 'Owner', 'Value': 'LeBaTam'}
        ]
    }
)
print("[THÀNH CÔNG] Đã gắn Tag Project và Owner cho S3 Bucket.")

In [ ]:
# 4. Tải dữ liệu ảnh thật PlantVillage (nếu chưa có) và tiền xử lý upload lên S3
import sys, os

# Đảm bảo môi trường Jupyter cài đủ thư viện
!{sys.executable} -m pip install -q pillow boto3 numpy

# Tự động tải bộ ảnh thật PlantVillage (Corn) nếu máy ảo chưa có dữ liệu
if not os.path.exists('Cloud/data') and not os.path.exists('../data') and not os.path.exists('data'):
    print('[INFO] Đang tải bộ dữ liệu ảnh thật PlantVillage (Corn)...')
    !git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git /tmp/plantvillage
    !mkdir -p Cloud/data
    !cp -r /tmp/plantvillage/raw/color/Corn* Cloud/data/

# Tự động tìm đường dẫn script và thư mục data phù hợp
script_candidates = ['Cloud/src/prepare_data.py', '../src/prepare_data.py', 'src/prepare_data.py']
script_path = next((p for p in script_candidates if os.path.exists(p)), 'Cloud/src/prepare_data.py')

data_candidates = ['Cloud/data', '../data', 'data']
data_path = next((p for p in data_candidates if os.path.exists(p)), 'Cloud/data')

print(f'Sử dụng script: {script_path}')
print(f'Nguồn dữ liệu : {data_path}')

!{sys.executable} {script_path} --source_dir {data_path} --s3_bucket {bucket_name} --max_total_images 1600


In [ ]:
# 5. Kiểm tra các thư mục kênh dữ liệu đã xuất hiện trên S3
response = s3.list_objects_v2(Bucket=bucket_name, Prefix='dataset/', MaxKeys=10)
print("Các đối tượng mẫu trên S3:")
for item in response.get('Contents', []):
    print(" -", item['Key'])